# Estudo passo a passo do pipeline

**Previsão de Movimentos de Ativos Financeiros com Machine Learning**

Este notebook não substitui os scripts. Ele **usa as mesmas funções** deles, uma etapa por vez, para mostrar os dados intermediários e as decisões de cada etapa. As células devem ser executadas em ordem.

> Pré-requisito: executar `python atualizar.py` pelo menos uma vez. O notebook lê os dados e os resultados gerados.

Roteiro:
1. Configuração do experimento
2. Dados brutos
3. Variáveis e alvo
4. Walk-forward e os modelos
5. Resultados

In [ ]:
import importlib

import numpy as np
import pandas as pd
from IPython.display import Image, display

import config

# os nomes dos scripts começam com número, por isso o import é feito assim
prep = importlib.import_module("02_preparar_variaveis")
modelar = importlib.import_module("03_modelar")

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

## 1. Configuração do experimento (`config.py`)

O `config.py` não executa nada: ele reúne os parâmetros do experimento (amostra, datas, variáveis, modelos e validação) num lugar só.

In [ ]:
pd.DataFrame(
    [(t, n, "ETF") for t, n in config.ETFS.items()]
    + [(t, n, "Ação") for t, n in config.ACOES.items()]
    + [(t, n, "FII") for t, n in config.FIIS.items()],
    columns=["ticker", "nome", "tipo"],
)

In [ ]:
pd.Series({
    "Início do download (aquecimento dos indicadores)": config.DATA_INICIO_DOWNLOAD,
    "Início da amostra analisada": config.DATA_INICIO,
    "Fim da amostra": config.DATA_FIM,
    "Início do período reservado": config.INICIO_RESERVADO,
    "Horizonte da previsão (pregões)": config.HORIZONTE,
    "Treino inicial (pregões)": config.TREINO_INICIAL,
    "Tamanho de cada teste (pregões)": config.JANELA_TESTE,
    "Modelos": ", ".join(config.MODELOS),
    "Variáveis": ", ".join(config.VARIAVEIS_TECNICAS),
}, name="valor").to_frame()

**Por que o download começa em 2017 se a análise começa em 2018?**

Alguns indicadores só existem depois de acumular histórico. O CCI precisa de 20 dias, o TRIX de cerca de 60 (três médias de 20) e o SMI de algumas dezenas. O ano extra serve de aquecimento: em 01/01/2018 todos os indicadores já têm valor.

## 2. Dados brutos (`01_baixar_dados.py`)

O script 01 baixa os 10 ativos, o VIX e o Ibovespa **de forma concorrente**, com `concurrent.futures.ThreadPoolExecutor`. Enquanto um download espera a resposta do servidor, outro pode avançar. Como essa tarefa é limitada pela rede e não pela CPU, threads são adequadas.

Dados baixados:

In [ ]:
pd.read_csv(config.DADOS_BRUTOS / "relatorio_download.csv")

In [ ]:
bova = prep.ler_precos("BOVA11")
print(bova.shape)          # (linhas = pregões, colunas)
bova.head()

In [ ]:
bova.describe().round(2)

In [ ]:
bova["fechamento"].plot(title="BOVA11 - fechamento ajustado", figsize=(10, 3));

**O que olhar:**
- Cada linha é um pregão. As colunas são abertura, máxima, mínima, fechamento e volume (OHLCV).
- O preço é **ajustado** (`auto_adjust=True`), ou seja, corrigido por dividendos e desdobramentos. Sem isso, um desdobramento pareceria uma queda de 50%.
- O CDI vem do Banco Central e é usado só no backtest, como o rendimento de "ficar no caixa":

In [ ]:
cdi = pd.read_csv(config.DADOS_BRUTOS / "cdi.csv", parse_dates=["data"])
cdi.tail()

## 3. Variáveis e alvo (`02_preparar_variaveis.py`)

### 3.1 Tratamento
A função `tratar` remove datas duplicadas e preços não positivos, corrige máxima e mínima inconsistentes e descarta variações diárias acima de 50%, tratadas como erro de dado ou de ajuste.

In [ ]:
px = prep.tratar(bova)
print(len(bova), "->", len(px), "pregões após a limpeza")

### 3.2 As 10 variáveis explicativas
`calcular_variaveis` recebe os preços e a variação do VIX e devolve uma linha por dia com todas as variáveis. As definições estão nas funções de `02_preparar_variaveis.py`.

In [ ]:
vix = prep.alinhar_vix(prep.variacao_vix(), px.index)
ind = prep.calcular_variaveis(px, vix)
ind[["preco"] + config.VARIAVEIS_TECNICAS].loc[config.DATA_INICIO:].head(10).round(3)

In [ ]:
ind[config.VARIAVEIS_TECNICAS].loc[config.DATA_INICIO:].describe().round(2)

| Variável | O que mede | Faixa típica |
|---|---|---|
| `vix_var` | variação do VIX no último pregão americano anterior | próximo de 0 |
| `cci` | distância do preço típico à sua média de 20 dias | -200 a +200 |
| `macd` | MACD normalizado (PPO): médias exponenciais de 12 e 26, em % | perto de 0 |
| `williams_r` | posição do fechamento entre a máxima e a mínima de 14 dias | -100 a 0 |
| `stoch_smi` | Stochastic Momentum Index (13, 2, 25) | -100 a +100 |
| `trix` | variação de uma média exponencial tripla, em % | perto de 0 |
| `bollinger_pctb` | posição dentro das Bandas de Bollinger | 0 a 1 (às vezes fora) |
| `cmo` | força das altas contra as baixas em 14 dias | -100 a +100 |
| `dpo` | preço sem a tendência, versão causal | perto de 0 |
| `roc` | retorno do dia | perto de 0 |

### 3.3 O alvo
`alvo = 1` se o fechamento ajustado do pregão seguinte for maior que o do pregão atual, e `0` caso contrário. O `ret_futuro` existe **só** para construir o alvo e calcular o backtest; ele não entra como variável explicativa.

In [ ]:
dados = pd.read_parquet(config.DADOS_PROCESSADOS / "dataset.parquet")
b = dados[dados["ativo"] == "BOVA11"].set_index("data")
b[["preco", "roc", "ret_futuro", "alvo"]].head(8)

Repare que o `ret_futuro` de um dia é igual ao `roc` **do dia seguinte**. É o mesmo número visto de dois momentos diferentes: hoje ele é desconhecido (é o que queremos prever) e amanhã ele vira informação conhecida.

### 3.4 Verificação de que as variáveis não usam o futuro
O vazamento temporal é um problema frequente em previsão financeira. A verificação: calcular as variáveis **cortando a série num dia D**, como se os dados posteriores não existissem, e comparar com o cálculo feito com a série inteira. Se os valores do dia D coincidem, as variáveis desse dia não dependeram de dados posteriores a D. A verificação vale para as datas testadas.

In [ ]:
dia = ind.loc[config.DATA_INICIO:].index[300]
ind_cortado = prep.calcular_variaveis(px.loc[:dia], vix.loc[:dia])
iguais = np.allclose(ind.loc[dia, config.VARIAVEIS_TECNICAS].astype(float),
                     ind_cortado.loc[dia, config.VARIAVEIS_TECNICAS].astype(float), equal_nan=True)
print(f"Dia testado: {dia.date()} -> indicadores iguais com e sem o futuro? {iguais}")

**Medidas adotadas para reduzir o vazamento temporal:**
1. as variáveis são calculadas com dados até o fechamento do dia da previsão, o que o teste acima confere em várias datas;
2. o VIX entra com um dia de defasagem, porque o mercado americano fecha depois da B3;
3. o DPO usa a versão causal, sem deslocar a média para o futuro;
4. no walk-forward, cada janela de teste é prevista por um modelo treinado apenas com pregões anteriores a ela.

Essas verificações não cobrem tudo: a escolha dos ativos usou critérios de histórico e liquidez avaliados no período inteiro, o que introduz viés de sobrevivência.

In [ ]:
print("Observações por ativo:")
display(dados.groupby(["tipo", "ativo"]).size().rename("dias").to_frame())
print("Proporção de dias de alta (alvo = 1):", round(dados["alvo"].mean(), 3))

A proporção de altas fica perto de 50%. Por isso uma taxa de acerto próxima de 50% não indica capacidade preditiva, e a comparação relevante é com referências simples, como prever sempre alta.

## 4. Walk-forward e os modelos (`03_modelar.py`)

### 4.1 As janelas
No walk-forward, o modelo é treinado com dados passados e avaliado no período seguinte, avançando no tempo. O primeiro treino usa cerca de 2 anos e o teste, os 6 meses seguintes. Depois, esses 6 meses entram no treino e os 6 meses seguintes são testados, e assim por diante (janela expansiva). A ordem temporal é preservada.

In [ ]:
b = dados[dados["ativo"] == "BOVA11"].sort_values("data").reset_index(drop=True)
janelas = []
for fold, tr, te in modelar.walk_forward(len(b)):
    janelas.append({"fold": fold,
                    "treino": f"{b.loc[tr[0], 'data'].date()} a {b.loc[tr[-1], 'data'].date()}",
                    "dias treino": len(tr),
                    "teste": f"{b.loc[te[0], 'data'].date()} a {b.loc[te[-1], 'data'].date()}",
                    "dias teste": len(te)})
pd.DataFrame(janelas)

**Por que não usar validação cruzada comum (k-fold)?**

No k-fold, a série é dividida em k blocos e cada bloco é testado com um modelo treinado nos demais. O problema não é apenas embaralhar os dias: mesmo sem embaralhar, quando o bloco de teste é, por exemplo, 2020, o treino inclui 2021 a 2026, ou seja, dados posteriores ao período testado. Na prática, essa informação não estaria disponível na data da previsão. Além disso, dias vizinhos são dependentes (os indicadores usam janelas que se sobrepõem), então pregões de treino colados ao bloco de teste carregam informação dele. As duas coisas tendem a inflar a acurácia.

O walk-forward evita isso: cada janela de teste é prevista apenas com pregões anteriores a ela, como aconteceria no uso real.

### 4.2 Uma janela, passo a passo
O script repete este processo para todas as janelas e ativos. Aqui está apenas a primeira janela do BOVA11.

In [ ]:
fold, tr, te = next(modelar.walk_forward(len(b)))
X = b[config.VARIAVEIS_TECNICAS].values
y = b["alvo"].astype(int).values

logistica = modelar.novo_modelo("logistica").fit(X[tr], y[tr])
floresta = modelar.novo_modelo("random_forest").fit(X[tr], y[tr])

resultado = b.loc[te, ["data", "alvo"]].copy()
resultado["prob_logistica"] = logistica.predict_proba(X[te])[:, 1]
resultado["prob_rf"] = floresta.predict_proba(X[te])[:, 1]
resultado["pred_logistica"] = (resultado["prob_logistica"] >= 0.5).astype(int)
resultado["pred_rf"] = (resultado["prob_rf"] >= 0.5).astype(int)
resultado.head(10).round(3)

In [ ]:
print("Acurácia neste fold:")
print("  Regressão Logística:", round((resultado["pred_logistica"] == resultado["alvo"]).mean(), 3))
print("  Random Forest:      ", round((resultado["pred_rf"] == resultado["alvo"]).mean(), 3))
print("  Sempre alta:        ", round((resultado["alvo"] == 1).mean(), 3))

**Como cada modelo decide:**
- A **Regressão Logística** atribui um peso a cada variável, combina-as linearmente e transforma o resultado numa probabilidade. Como as variáveis foram padronizadas, os coeficientes podem ser comparados entre si.
- A **Random Forest** combina 300 árvores de decisão, cada uma com divisões do tipo "o CCI está acima de X?", e agrega os votos. A importância de cada variável mostra quanto ela contribuiu, em média, para separar altas de baixas.

In [ ]:
pd.DataFrame({
    "peso_logistica": logistica[-1].coef_[0],
    "importancia_rf": floresta.feature_importances_,
}, index=config.VARIAVEIS_TECNICAS).sort_values("importancia_rf", ascending=False).round(3)

## 5. Resultados (`resultados/`)

A partir daqui são usados os resultados de **todas as janelas de todos os ativos**, gerados pelo script 03.

In [ ]:
resumo = pd.read_csv(config.RESULTADOS / "metricas_resumo.csv")
resumo[resumo["segmento"].isin(["geral", "periodo"])][
    ["segmento", "valor", "modelo", "n", "acuracia", "auc", "f1", "p_valor_vs_sempre_alta"]
].round(3)

**Como ler:**
- **acuracia:** fração de dias em que o modelo acertou a direção.
- **auc:** capacidade de separar altas de baixas pela probabilidade. 0,5 equivale a uma ordenação ao acaso.
- **p_valor_vs_sempre_alta:** supondo que a taxa de acerto verdadeira do modelo fosse igual à de "sempre alta" (hipótese nula), é a probabilidade de observar um número de acertos igual ou maior que o obtido. Abaixo de 0,05, rejeita-se a hipótese nula e a vantagem é considerada estatisticamente significativa. O p-valor **não** é a probabilidade de o modelo ter vencido por sorte, nem a probabilidade de a hipótese nula ser verdadeira.
- **período reservado:** o último ano, em que nenhuma decisão do trabalho foi tomada; serve como confirmação adicional.

In [ ]:
comp = pd.read_csv(config.RESULTADOS / "comparacao_modelos.csv")
comp[comp["segmento"].isin(["geral", "periodo", "ano"])][
    ["segmento", "valor", "acuracia_a", "acuracia_b", "so_a_acerta", "so_b_acerta", "p_valor_mcnemar"]
].rename(columns={"acuracia_a": "acc_logistica", "acuracia_b": "acc_rf",
                  "so_a_acerta": "so_logistica_acerta", "so_b_acerta": "so_rf_acerta"}).round(4)

**Teste de McNemar.** O teste considera só os dias em que **um modelo acertou e o outro errou**. Se os dois modelos tivessem o mesmo desempenho (hipótese nula), esses dias se dividiriam meio a meio entre eles. O p-valor é a probabilidade de observar uma divisão tão desigual quanto a obtida, ou mais, se a hipótese nula fosse verdadeira. Abaixo de 0,05, a diferença entre os modelos é considerada estatisticamente significativa.

A linha "ano" mostra se a diferença se mantém ao longo do tempo.

In [ ]:
pd.read_csv(config.RESULTADOS / "estabilidade.csv")

In [ ]:
por_ano = pd.read_csv(config.RESULTADOS / "metricas_por_ano.csv")
tab = por_ano.pivot(index="ano", columns="modelo", values="acuracia")
tab.round(3)

In [ ]:
for fig in sorted(config.FIGURAS.glob("*.png")):
    print(fig.name)
    display(Image(filename=str(fig), width=700))